# PAWS Project 2 — experiments and analysis

Pair classification with pretrained embeddings (`stack_diff_prod`).
Logic lives in `part2/project/*.py`. This notebook is mainly for **analysis**.

Training is off by default — flip the toggles below when you are on the GPU machine.

## Setup and toggles

In [ ]:
# Flip these when you intentionally want heavy work.
RUN_TRAINING = True          # full enabled RUNS from config (GPU)
RUN_DEVTEST_PREDS = True      # load models and score devtest (needs trained checkpoints)
RUN_BIN_PLOTS = True
RUN_GOTCHA = True
RUN_LENGTH_X_MODEL = True
RUN_BDA = True                # MCMC — can be slow

# Systems compared in analysis (must match completed run ids under results/)
ANALYSIS_SYSTEMS = [
    "pool_ft300",
    "lstm_ft300",
    "bilstm_ft300",
]

# Which seed/fold checkpoints to use for devtest predictions
ANALYSIS_SEED = 0
ANALYSIS_FOLD = 0

# Embedding used to tokenize analysis features / reload models that share vocab
# (pool/lstm/bilstm ft300 all use fasttext300)
ANALYSIS_EMB_KEY = "fasttext300"

In [2]:
import sys
from pathlib import Path

# Notebook lives in part2/project/; repo root is two levels up.
root = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(root))
print("repo root:", root)

import json

import numpy as np
import pandas as pd
import torch

from part1.baseline import data as paws_data
from part2.project import analysis, config, embeddings, experiment

print("device:", config.DEVICE)
print("results:", config.RESULTS_DIR)
print("enabled runs:", [r["id"] for r in config.enabled_runs()])

repo root: C:\Users\joeyl\Documents\GitHub\nlp1-student
device: cpu
results: C:\Users\joeyl\Documents\GitHub\nlp1-student\part2\project\results
enabled runs: ['pool_ft300', 'lstm_ft300', 'lstm_glove300', 'bilstm_ft300', 'bilstm_glove300']


## Data, embeddings, tokenizer

In [3]:
outcome_space, cv_splits, devtest_splits = experiment.load_paws_cv_and_devtest()
print("outcome space:", outcome_space)
print("CV seeds:", config.SEEDS, "folds:", config.NUM_FOLDS)
print("devtest size (seed 0):", len(devtest_splits[0]["devtest"]))

emb_model, tokenizer = embeddings.load_embedding_and_tokenizer(ANALYSIS_EMB_KEY)
print("embedding:", ANALYSIS_EMB_KEY, emb_model.vectors.shape)

outcome space: 0: not paraphrase, 1: paraphrase
CV seeds: [0, 3, 5] folds: 5
devtest size (seed 0): 200
embedding: fasttext300 (830463, 300)


## Training (optional)

Default is skip. On the Linux GPU machine set `RUN_TRAINING = True`, or run from the shell:

```bash
python -m part2.project.experiment
python -m part2.project.experiment --run-id pool_ft300
```

In [4]:
if RUN_TRAINING:
    summaries, _ = experiment.run_enabled(device=config.DEVICE)
    display(pd.DataFrame(summaries))
else:
    print("Skipping training (RUN_TRAINING=False).")
    summary_paths = list(config.RESULTS_DIR.glob("*/summary.json"))
    if summary_paths:
        rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(summary_paths)]
        display(pd.DataFrame(rows))
    else:
        print("No summary.json files found yet under", config.RESULTS_DIR)

Skipping training (RUN_TRAINING=False).
No summary.json files found yet under C:\Users\joeyl\Documents\GitHub\nlp1-student\part2\project\results


## Devtest predictions for analysis

Uses one seed/fold checkpoint per system (see toggles). Needs trained `model.pt` files.

In [5]:
seed_idx = config.SEEDS.index(ANALYSIS_SEED)
devtest_pairs = devtest_splits[seed_idx]["devtest"]
devtest_xy = [(s1, s2) for (s1, s2), _y in devtest_pairs]
devtest_labels = [int(y) for _x, y in devtest_pairs]

system_predictions = {}

if RUN_DEVTEST_PREDS:
    for run_id in ANALYSIS_SYSTEMS:
        model_path = config.RESULTS_DIR / run_id / f"seed-{ANALYSIS_SEED}" / f"fold-{ANALYSIS_FOLD}" / "model"
        if not Path(f"{model_path}.pt").is_file():
            print(f"Missing checkpoint: {model_path}.pt — skip {run_id}")
            continue
        # Reload emb matching this run (glove vs fasttext)
        run = config.run_by_id(run_id)
        run_emb, run_tok = embeddings.load_embedding_and_tokenizer(run["emb"])
        model = experiment.load_model(str(model_path), run_emb, device=config.DEVICE)
        logprobs = experiment.predict_logprobs(
            model, run_tok, devtest_xy, batch_size=config.OPT_KWARGS["batch_size"], device=config.DEVICE
        )
        system_predictions[run_id] = logprobs.argmax(axis=-1)
        print(run_id, "devtest acc",
              float(np.mean(system_predictions[run_id] == np.asarray(devtest_labels))))
else:
    print("Skipping devtest predictions (RUN_DEVTEST_PREDS=False).")

if system_predictions:
    # Features/bins use ANALYSIS_EMB tokenizer for avg_len_tokens
    analysis_df = analysis.build_analysis_frame(devtest_pairs, system_predictions, tokenizer)
    display(analysis_df.head())
else:
    analysis_df = None
    print("No predictions available — analysis plots below will skip.")

Missing checkpoint: C:\Users\joeyl\Documents\GitHub\nlp1-student\part2\project\results\pool_ft300\seed-0\fold-0\model.pt — skip pool_ft300
Missing checkpoint: C:\Users\joeyl\Documents\GitHub\nlp1-student\part2\project\results\lstm_ft300\seed-0\fold-0\model.pt — skip lstm_ft300
Missing checkpoint: C:\Users\joeyl\Documents\GitHub\nlp1-student\part2\project\results\bilstm_ft300\seed-0\fold-0\model.pt — skip bilstm_ft300
No predictions available — analysis plots below will skip.


## Accuracy vs feature bins

Compare pool vs LSTM vs BiLSTM on the same devtest pairs.

In [6]:
if RUN_BIN_PLOTS and analysis_df is not None:
    analysis.plot_accuracy_vs_bins(
        analysis_df, "avg_len_words_bin", title="Accuracy vs avg length (words)"
    )
    analysis.plot_accuracy_vs_bins(
        analysis_df, "avg_len_tokens_bin", title="Accuracy vs avg length (tokens)"
    )
    analysis.plot_accuracy_vs_bins(
        analysis_df, "overlap_bin", title="Accuracy vs word overlap"
    )
else:
    print("Skipping bin plots.")

Skipping bin plots.


## PAWS gotcha: high-overlap × non-paraphrase error rate

In [7]:
if RUN_GOTCHA and analysis_df is not None:
    gotcha_df, threshold = analysis.high_overlap_non_paraphrase_error_rate(analysis_df)
    print(f"High-overlap threshold (q={config.OVERLAP_HIGH_QUANTILE}): {threshold:.4f}")
    display(gotcha_df)
else:
    print("Skipping gotcha table.")

Skipping gotcha table.


## Length × model: uni-LSTM vs BiLSTM

In [8]:
if RUN_LENGTH_X_MODEL and analysis_df is not None:
    length_table = analysis.length_by_model_table(
        analysis_df,
        length_bin_col="avg_len_words_bin",
        systems=("lstm_ft300", "bilstm_ft300"),
    )
    display(length_table)
else:
    print("Skipping length × model table.")

Skipping length × model table.


## BDA (correctness ~ system + features)

Uses `part2.analysis.fit_bda_model` via `analysis.fit_bda_from_frame`.

In [9]:
if RUN_BDA and analysis_df is not None:
    import arviz as az
    from part2 import analysis as part2_analysis

    mcmc, system_names, feature_cols = analysis.fit_bda_from_frame(analysis_df)
    # pred one-hot (K) + continuous feature_cols — must match fit_bda_from_frame
    feature_names = [f"pred_{k}" for k in range(len(outcome_space))] + list(feature_cols)
    idata = part2_analysis.posterior_to_arviz(mcmc, system_names, feature_names)
    print("systems:", system_names)
    print("features:", feature_names)
    display(az.summary(idata, var_names=["sys_shift", "accuracy"]))
else:
    print("Skipping BDA.")

Skipping BDA.
